# Preisvorhersage mit einer multiplen linearen Regression

## 1. Einleitung

Dieses Notebook knüpft an die explorative Analyse und das Regressionsbaum-Notebook an und erstellt mit der multiplen linearen Regression ein zweites Vorhersagemodell. Ziel ist es erneut, den Angebotspreis eines Gebrauchtwagens anhand der Fahrzeugmerkmale zu schätzen und die zweite Forschungsfrage zu beantworten. Durch den Einsatz eines zweiten Verfahrens lässt sich zusätzlich beurteilen, wie robust die Ergebnisse gegenüber der Wahl des Modells sind.

Die multiple lineare Regression wurde gewählt, weil sie den Einfluss jedes einzelnen Merkmals als Koeffizient ausdrückt und dadurch besonders gut interpretierbar ist. Sie ergänzt den Regressionsbaum, der zwar flexible Zusammenhänge abbildet, seine Wirkung je Merkmal aber weniger transparent macht.

Ein zentraler Punkt ist der Umgang mit der starken Rechtsschiefe der Preise, die sich bereits in der explorativen Analyse und in der Bewertung des Regressionsbaums gezeigt hat. Da wenige sehr teure Fahrzeuge die Ergebnisse verzerren, wird der Preis logarithmisch transformiert, wodurch die extremen Werte gestaucht werden und die Annahmen der linearen Regression besser erfüllt sind. Als Vergleichsmaßstab dient wieder ein einfaches Baseline-Modell. Die Güte wird mit den Kennzahlen R², RMSE und MAE bewertet, durch eine Analyse der Residuen überprüft und mit einer Kreuzvalidierung abgesichert.

## 2. Setup

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from scipy import stats

# Darstellung
%matplotlib inline
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# Pfade, eigener Bilder-Ordner für dieses Notebook
DATA_PROCESSED = Path("../data/processed/data_processed.csv")
IMG_DIR = Path("../images/modeling/regression")
IMG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Bereinigte Daten laden
df = pd.read_csv(DATA_PROCESSED)
print("Shape:", df.shape)
df.head()

## 3. Merkmalsauswahl und Vorbereitung

In [ ]:
# Zielgröße und Merkmale, identisch zum Regressionsbaum für die Vergleichbarkeit
ziel = "price_in_euro"
num_features = ["year", "mileage_in_km", "power_ps"]
kat_features = ["brand", "color", "transmission_type", "fuel_type"]

X = df[num_features + kat_features]
y = df[ziel]

print("Merkmale X:", X.shape, "Zielgröße y:", y.shape)

In [ ]:
# Vergleich der Verteilung von Preis und logarithmiertem Preis
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].hist(y, bins=60, range=(0, 100000), color="#4C72B0", edgecolor="white")
axes[0].set_title("Preis in Euro")
axes[0].set_xlabel("Preis")
axes[0].set_ylabel("Anzahl Fahrzeuge")

axes[1].hist(np.log1p(y), bins=60, color="#4C72B0", edgecolor="white")
axes[1].set_title("Logarithmierter Preis")
axes[1].set_xlabel("log(1 + Preis)")

plt.tight_layout()
plt.savefig(IMG_DIR / "log_transformation.png", dpi=150, bbox_inches="tight")
plt.show()

Der Preis ist im Original stark rechtsschief, während der logarithmierte Preis eine annähernd symmetrische Verteilung aufweist. Die Modellierung auf der logarithmischen Skala erfüllt die Annahmen der linearen Regression daher besser und verringert den Einfluss der wenigen sehr teuren Fahrzeuge.

## 4. Train-Test-Split und Encoding

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Training:", X_train.shape, "Test:", X_test.shape)

In [ ]:
# Vorverarbeitung: drop="first" lässt je kategorialem Merkmal
# die erste Kategorie als Referenzkategorie weg
preprocessor = ColumnTransformer(
    transformers=[
        (
            "kat",
            OneHotEncoder(drop="first", handle_unknown="ignore"),
            kat_features
        )
    ],
    remainder="passthrough"
)

# Vorverarbeitung an die Trainingsdaten anpassen
preprocessor.fit(X_train)

# Anzahl der Merkmale nach der Kodierung
anzahl_merkmale = preprocessor.get_feature_names_out().shape[0]

# Trainierten OneHotEncoder auslesen
encoder = preprocessor.named_transformers_["kat"]

# Ausgelassene Referenzkategorien bestimmen
referenz_tabelle = pd.DataFrame({
    "Merkmal": kat_features,
    "Ausgelassene Referenzkategorie": [
        kategorien[drop_index]
        for kategorien, drop_index
        in zip(encoder.categories_, encoder.drop_idx_)
    ]
})

# Gemeinsame Ausgabe
print(f"Anzahl Merkmale nach Kodierung: {anzahl_merkmale}")
print("\nAusgelassene Referenzkategorien:")

display(referenz_tabelle)

Die Daten wurden in 194.448 Trainings- und 48.612 Testfälle aufgeteilt. Mit drop="first" wurde je kategorialem Merkmal eine Referenzkategorie ausgeschlossen. Nach der Kodierung umfasst das Modell 73 Merkmale; die Referenzen sind in der Tabelle ausgewiesen.

## 5. Baseline

In [ ]:
# Baseline, identisch zum Baum, sagt immer den mittleren Trainingspreis vorher
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train[num_features], y_train)
print("Mittlerer Trainingspreis:", round(y_train.mean(), 2), "Euro")

Das Baseline-Modell sagt unabhängig von den Merkmalen für jedes Fahrzeug den mittleren Trainingspreis von rund 26142 Euro vorher. Es ist bewusst dasselbe Vergleichsmodell wie im Regressionsbaum, sodass beide Verfahren am identischen Maßstab gemessen werden. Die multiple lineare Regression muss diesen Wert deutlich unterbieten, damit sich ihr Einsatz rechtfertigt.

## 6. Multiple lineare Regression

In [ ]:
# Pipeline aus Vorverarbeitung und linearer Regression auf logarithmiertem Preis
regression = Pipeline(steps=[
    ("vorverarbeitung", preprocessor),
    ("modell", TransformedTargetRegressor(
        regressor=LinearRegression(),
        func=np.log1p,
        inverse_func=np.expm1
    ))
])

# Training ausschließlich auf den Trainingsdaten
regression.fit(X_train, y_train)

In [ ]:
# Trainiertes Regressionsmodell und Merkmalsnamen aus der Pipeline holen
lin = regression.named_steps["modell"].regressor_
feature_names = regression.named_steps["vorverarbeitung"].get_feature_names_out()

# Koeffizienten sammeln und Namen aufräumen
koef = pd.Series(lin.coef_, index=feature_names)
koef.index = koef.index.str.replace("remainder__", "").str.replace("kat__", "")

# Da auf der log-Skala trainiert wurde, als prozentualer Effekt lesbar
effekt = (np.exp(koef) - 1) * 100

koef_tabelle = pd.DataFrame({
    "Koeffizient (log)": koef.round(5),
    "Effekt (%) je Einheit": effekt.round(3)
}).sort_values("Effekt (%) je Einheit", ascending=False)

koef_tabelle

In [ ]:
# Numerische Merkmale in sinnvollen Einheiten
for merkmal, einheit, faktor in [
    ("year", "pro Jahr", 1),
    ("mileage_in_km", "pro 10.000 km", 10000),
    ("power_ps", "pro 10 PS", 10),
]:
    b = koef[merkmal] * faktor
    print(f"{merkmal:16s} {einheit:16s} {(np.exp(b)-1)*100:+.1f} %")

Die multiple lineare Regression macht den Einfluss jedes Merkmals als prozentualen Effekt auf den Preis sichtbar. Ein um ein Jahr neueres Baujahr erhöht den geschätzten Preis um rund 7,4 Prozent, je zusätzliche 10.000 Kilometer sinkt er um rund 2,9 Prozent, und je zusätzliche 10 PS steigt er um rund 4,5 Prozent. Bei den kategorialen Merkmalen sind Dieselfahrzeuge gegenüber der Referenz rund 14,8 Prozent teurer und Benzinfahrzeuge rund 12,1 Prozent günstiger, ein Schaltgetriebe senkt den Preis gegenüber der Automatik um rund 11 Prozent, und Premiummarken wie Mercedes-Benz liegen über den Massenmarken. Die Richtung aller Effekte entspricht den Erwartungen und bestätigt die Ergebnisse aus der explorativen Analyse und dem Regressionsbaum.

In [ ]:
# Residuen auf der log-Skala, auf der das Modell trainiert wurde
log_wahr = np.log1p(y_test)
y_pred_regression = regression.predict(X_test)
log_vorhersage = np.log1p(y_pred_regression)
residuen = log_wahr - log_vorhersage

# Für eine übersichtliche Darstellung eine Stichprobe
stichprobe = pd.DataFrame(
    {"vorhersage": log_vorhersage, "residuum": residuen}
).sample(8000, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Residuen gegen angepasste Werte
axes[0].scatter(stichprobe["vorhersage"], stichprobe["residuum"],
                alpha=0.15, s=8, color="#4C72B0")
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_title("Residuen und angepasste Werte")
axes[0].set_xlabel("Vorhergesagter log-Preis")
axes[0].set_ylabel("Residuum")

# Q-Q-Plot der Residuen
stats.probplot(stichprobe["residuum"], dist="norm", plot=axes[1])
axes[1].set_title("Q-Q-Plot der Residuen")

plt.tight_layout()
plt.savefig(IMG_DIR / "residuen.png", dpi=150, bbox_inches="tight")
plt.show()

Die Residuen streuen weitgehend gleichmäßig um die Nulllinie, sodass Linearität und konstante Fehlervarianz im mittleren Preisbereich weitgehend erfüllt sind. Der Q-Q-Plot zeigt jedoch schwerere Ränder als eine Normalverteilung, weshalb sehr günstige und sehr teure Fahrzeuge mit größeren Fehlern geschätzt werden. Die logarithmische Transformation verbessert die Modellannahmen deutlich, hebt die Abweichungen an den Extremen aber nicht vollständig auf.

## 7. Bewertung und Robustheit

In [ ]:
# Vorhersagen auf den Testdaten
y_pred_baseline = baseline.predict(X_test[num_features])
y_pred_regression = regression.predict(X_test)

# Hilfsfunktion für die drei Kennzahlen
def kennzahlen(y_wahr, y_vorhersage):
    r2 = r2_score(y_wahr, y_vorhersage)
    rmse = mean_squared_error(y_wahr, y_vorhersage) ** 0.5
    mae = mean_absolute_error(y_wahr, y_vorhersage)
    return r2, rmse, mae

r2_b, rmse_b, mae_b = kennzahlen(y_test, y_pred_baseline)
r2_r, rmse_r, mae_r = kennzahlen(y_test, y_pred_regression)

# Ergebnisse gegenüberstellen
ergebnisse = pd.DataFrame({
    "Baseline": [r2_b, rmse_b, mae_b],
    "Lineare Regression": [r2_r, rmse_r, mae_r],
}, index=["R²", "RMSE (Euro)", "MAE (Euro)"])

ergebnisse.round(2)

Die lineare Regression übertrifft die Baseline deutlich und senkt den mittleren absoluten Fehler von rund 15.529 auf rund 6.198 Euro bei einem R² von 0,60. Der Regressionsbaum war auf demselben Testfeld mit einem R² von 0,79 und einem MAE von rund 5.154 Euro etwas genauer, da er nichtlineare Zusammenhänge flexibler abbildet. Die Regression bietet dafür zwei Vorteile, nämlich die unmittelbare Interpretierbarkeit über ihre Koeffizienten und eine durch die logarithmische Skala erhöhte Robustheit gegenüber den Extrempreisen.

In [ ]:
# Robustheit über Kreuzvalidierung, R² und MAE über fünf Durchläufe
kf = KFold(n_splits=5, shuffle=True, random_state=42)

r2_cv = cross_val_score(regression, X_train, y_train, cv=kf, scoring="r2")
mae_cv = -cross_val_score(regression, X_train, y_train, cv=kf, scoring="neg_mean_absolute_error")

print("R² je Durchlauf:", r2_cv.round(3))
print("R² Mittelwert:", r2_cv.mean().round(3), "Streuung:", r2_cv.std().round(3))
print()
print("MAE je Durchlauf (Euro):", mae_cv.round(0))
print("MAE Mittelwert (Euro):", round(mae_cv.mean()), "Streuung:", round(mae_cv.std()))

Die Kreuzvalidierung liefert ein R² zwischen 0,40 und 0,60, im Mittel 0,535 mit einer geringen Streuung von 0,07. Der MAE bleibt über alle fünf Durchläufe eng zwischen rund 6.372 und 6.618 Euro, im Mittel bei 6.479 Euro mit einer Streuung von nur 90 Euro. Beide Kennzahlen sind damit sehr stabil, sodass das Modell zuverlässig verallgemeinert.

## 8. Zwischenfazit


In diesem Notebook wurde mit der multiplen linearen Regression ein zweites Modell zur Preisvorhersage erstellt. Es nutzt dieselben Merkmale wie der Regressionsbaum, transformiert den Preis wegen der starken Rechtsschiefe logarithmisch und lässt bei der Kodierung je Merkmal eine Referenzkategorie weg, um Multikollinearität zu vermeiden. Der Vorteil des Verfahrens liegt in der Interpretierbarkeit, denn jeder Koeffizient beschreibt einen prozentualen Effekt auf den Preis. Ein neueres Baujahr, mehr Leistung sowie Diesel und Premiummarken erhöhen den geschätzten Preis, während eine höhere Laufleistung, ein Schaltgetriebe und Massenmarken ihn senken.

Auf dem einzelnen Testfeld erreicht die Regression ein R² von 0,60 und einen mittleren Fehler von rund 6.198 Euro und übertrifft die Baseline damit deutlich. Der Regressionsbaum war auf diesem Testfeld zwar etwas genauer, die Kreuzvalidierung zeigt jedoch, dass die Regression mit einem mittleren R² von 0,535 und einer Streuung von nur 0,07 deutlich stabiler verallgemeinert. Die Residuenanalyse bestätigt, dass die Modellannahmen im mittleren Preisbereich weitgehend erfüllt sind, an den Extremen aber schwerere Ränder auftreten. Insgesamt ergänzt die Regression den Baum, indem sie etwas an Genauigkeit einbüßt, dafür aber robuster und transparent interpretierbar ist.